# Lab 3 — Data Cleaning and Quality
## Think first. Inspect evidence. Explain your decision.
**Data Science Fundamentals · New Uzbekistan University · Mohammad Sabokrou**

**Time:** approximately 110 minutes + 10 minutes to check and submit.  
**Goal:** decide what data can support, not demonstrate programming skill.

All code is provided. Edit only your identity, the three decision settings, and written answers. No formulas to memorize or derive. Run cells in order using ▶ or Shift+Enter. In Colab, double-click a text cell to answer; use **File → Download → Download .ipynb** to save your completed notebook.

Your student ID generates a reproducible version: scores, missing responses, review cases, and challenge details vary. Use your own numbers and record IDs. A generic answer without evidence is incomplete. Discuss ideas, but explain your own reasoning.

**Learning outcomes:** diagnose quality problems; justify safe cleaning; distinguish missingness mechanisms; interpret sensitivity and outlier checks; report a result with its denominator, uncertainty, and audit trail.

| Part | Focus | Minutes | Points |
|---|---|---:|---:|
| 1 | Purpose, dictionary, inventory | 15 | 15 |
| 2 | Evidence-based cleaning decisions | 25 | 25 |
| 3 | Missingness and who remains | 20 | 20 |
| 4 | Missing-data choices and sensitivity | 15 | 15 |
| 5 | Unusual values and consequences | 15 | 10 |
| 6 | Audit, reporting, changed-case challenge | 20 | 15 |

Points reward reasoning, relevant evidence, and limitations. Code elegance and writing formulas receive no credit. Keep missing values unfilled in your final registry. The temporary imputation example is for comparison only.

## Setup — your version
Enter your real student ID as **text inside quotation marks**, preserving leading zeros. Enter your name and class group. The ID check verifies format only; it does not verify registration. The notebook starts without outputs. Replace the demonstration ID with yours, then run the cells in order.

In [5]:
STUDENT_ID = "250452"
STUDENT_NAME = "Xolmurodov Asilbek"
CLASS_GROUP = "SSE2"

import hashlib, re, io, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

assert re.fullmatch(r"[A-Za-z0-9_-]{3,30}", STUDENT_ID), "Use 3–30 letters, digits, _ or -; preserve leading zeros."
seed = int(hashlib.sha256(STUDENT_ID.encode()).hexdigest()[:8], 16)
rng = np.random.default_rng(seed)
print("Student:", STUDENT_ID, "| Version:", f"{seed:08x}")
if STUDENT_ID == "DEMO123":
    print("DEMONSTRATION ONLY — enter your own ID before submitting.")

Student: 250452 | Version: 0c32e3a6


## 1 · What does “clean” mean for this task?
You are preparing a **current registry of 12 fictional students**. The question is: *What entry scores are observed, and how complete is our information about this cohort?* These are synthetic teaching data, not university records.

**One final row = one current student.** A separate weekly-results table would have a different row definition.

| Field | Meaning and rule |
|---|---|
| student_id | Text identifier, unique in the final registry; preserve leading zeros |
| programme | Biology, Bioengineering, or Economics; the source confirms BIO means Biology |
| group | Day or Evening; known for every student |
| age | Years; dictionary accepts 16–100 for this fictional cohort, but validity does not prove accuracy |
| entry_score | Points out of 100. Here “80%” means 80 points; plain numbers already mean points |
| phone | Contact number; not required for score summaries |
| updated_at | Source revision date, explicitly day/month/year; newest does not automatically mean authoritative |

The expected roster contains exactly IDs 00101–00112. Blank strings represent missing values. Source notes will be supplied in Part 2.

In [6]:
ids = [f"{101+i:05d}" for i in range(12)]
scores = rng.integers(55, 91, 12)
older_age = int(rng.choice([43, 47, 52]))
base = pd.DataFrame({
    "student_id": ids,
    "programme": [" biology ", "BIO", "Biology", "Bioengineering", "Economics", "Economics"]*2,
    "group": ["Day"]*8 + ["Evening"]*4,
    "age": ["19", "20", "220", "21", "19", "22", "20", "21", str(older_age), "22", "21", "20"],
    "entry_score": [str(x) for x in scores],
    "phone": [f"555-{i:04d}" for i in range(12)],
    "updated_at": ["03/04/2026"]*12,
})
base.loc[3, "entry_score"] += "%"
base.loc[5, "entry_score"] = ""
base.loc[9, "entry_score"] = "seventy"
base.loc[10, "phone"] = ""
correction = base.iloc[[1]].copy()
correction["entry_score"] = str(min(int(scores[1])+7, 100))
correction["updated_at"] = "05/04/2026"
raw = pd.concat([base, base.iloc[[0]], correction], ignore_index=True)
raw.insert(0, "record_id", [f"R{i+1:02d}" for i in range(len(raw))])
raw_snapshot = raw.copy(deep=True)
display(raw)
print("Rows:", len(raw), "| Distinct students:", raw.student_id.nunique())
print("Expected roster IDs absent:", sorted(set(ids)-set(raw.student_id)))
print("Blank cells by field:")
display(raw.eq("").sum().rename("blank_cells").to_frame())

,record_id,student_id,programme,group,age,entry_score,phone,updated_at
0,R01,00101,biology,Day,19,87,555-0000,03/04/2026
1,R02,00102,BIO,Day,20,63,555-0001,03/04/2026
2,R03,00103,Biology,Day,220,79,555-0002,03/04/2026
3,R04,00104,Bioengineering,Day,21,63%,555-0003,03/04/2026
4,R05,00105,Economics,Day,19,84,555-0004,03/04/2026
5,R06,00106,Economics,Day,22,,555-0005,03/04/2026
6,R07,00107,biology,Day,20,64,555-0006,03/04/2026
7,R08,00108,BIO,Day,21,86,555-0007,03/04/2026
8,R09,00109,Biology,Evening,52,61,555-0008,03/04/2026
9,R10,00110,Bioengineering,Evening,22,seventy,555-0009,03/04/2026


Rows: 14 | Distinct students: 12
Expected roster IDs absent: []
Blank cells by field:


,blank_cells
record_id,0
student_id,0
programme,0
group,0
age,0
entry_score,1
phone,1
updated_at,0


**Q1 · Quality depends on purpose (8 points).**
1. Identify one issue for **each** of completeness, validity, consistency, uniqueness, accuracy, and timeliness. For dimensions not diagnosable here, state a concrete check or external evidence needed; do not invent an error.
2. Which missing field matters for urgent phone calls but not for the entry-score summary? Why should its blank not cause the whole row to be deleted?
3. Give the row count, distinct-student count, and absent-roster count in your version. Why can these three quantities differ?

**Write here:* 1.
 In terms of completeness, as seen in R06 the entry_score is missing which was a very important data, but missing contact number can also contribute to the completeness of the data.

 In terms of validity, only R03 is showing 220 as age which is far beyond that 16-100 acceptable range, the rest lies well on the range with only 52 a little different but still within the range.

 In terms of consistency, the inconsistent values can be seen in entry_score for R04 and R10 because the range to enter the data was from 16-100, and writing the score with % or letters create an invalid output. Another inconsistency can be seen in R02, because the data in it is already outdated with the new one.

 In terms of uniqueness, R13 is the duplicate of the first row, so except 00101, all are unique since they have no duplicates.

 In terms of acccuracy, the data's resource must be investigated and accuracy can be wrong on the age of student 00103 but the making sure is still necessart before decision and it can not be interpreted as inaccurate before investigation.

 In terms of timeliness, the only row worth attention is the last R14 because it shows 00102 student taking the second entry test after 2 days with different entry_score which might cause for the previous score to be updated with new one.

 2.

 R10 has the phone number not present and it is written in the definition of phone that it is not required for score summary but it will come in handy when urgent followups have to be done. The whole row shouldn't be deleted since the other important details are still available and score summary can also be done easily.

 3.

 Total rows are 14, Distinct students are 12, and there is no absent ID. They can differ because the student will take exam several times as seen in the case of 00102 or system might duplicate some of the students' data.




### Your explanation
**Q2 · Meaning before conversion (7 points).**
1. In this dictionary, what number should replace the percentage text? Why would automatically dividing it by 100 be wrong here?
2. What happens if identifiers become integers? Interpret 03/04/2026 using the documented source convention.
3. Explain why “seventy” is not the same situation as an originally blank score. Does a parser failure prove the intended value was zero?

**Write here:** 1.
The symbol % should be removed from the R04 and the enntry_score should simply be 63 instead of 63%. It is nonsense to divide by 100 which gives 0.63 and doesn't even lie in the range of 16-100.

2.
For student ID, changing it to integer will not preserve the leading 0s which was in the definition, for day/month/year format in the definition, this will be interpreted as April 3, 2026.

3.
seventy is present but invalid formatted input, while blank is completely missing.


## 2 · Choose actions before running the cleaning
**Evidence packet from the fictional source owner:**

- R13 is a confirmed extra export copy of R01.
- R14 is a verified correction of R02 for the same entry exam; it supersedes R02. This decision is based on verification, not just its date.
- BIO and outside spaces can safely be standardized to Biology. Bioengineering is a separate programme.
- The intended value for age 220 is unknown. The older student's age is verified correct.
- The entry “seventy” is not verified against the exam record. Flag the unparsed value and preserve the raw text; do not guess.
- No source measurement is available for the originally blank score.

**Predict before executing:** use the evidence to decide which rows to remove, which values to flag, and which values to retain.

### Your explanation
**Q3 · Your decision plan (12 points).** Complete the table. “It looks wrong” is not enough evidence.

| Record or field | Keep / standardize / remove copy / use correction / flag value | Evidence and reason | Risk of an alternative action |
|---|---|---|---|
| R13 | … | … | … |
| R02 and R14 | … | … | … |
| age 220 | … | … | … |
| verified older age | … | … | … |
| Bioengineering | … | … | … |
| “seventy” | … | … | … |

Predict final row count and number of missing entry scores: **…**

**Write here:** …

### Decision controls — change only the quoted choices
These settings let you inspect consequences without writing an algorithm. First choose your preferred policy. You may then experiment; return to a defensible policy before exporting.

- `COPY_POLICY`: `"remove_confirmed"` or `"keep_all"`
- `VERSION_POLICY`: `"verified_correction"` or `"keep_both"`
- `INVALID_AGE_POLICY`: `"flag_value"` or `"drop_student"`

The code keeps the original export unchanged. It never fills a missing entry score.

In [ ]:
COPY_POLICY = "remove_confirmed"
VERSION_POLICY = "verified_correction"
INVALID_AGE_POLICY = "flag_value"

assert COPY_POLICY in ["remove_confirmed", "keep_all"]
assert VERSION_POLICY in ["verified_correction", "keep_both"]
assert INVALID_AGE_POLICY in ["flag_value", "drop_student"]
clean = raw.copy(deep=True)
log = []
def record(issue, affected, unit, action, evidence):
    log.append(dict(issue=issue, affected=int(affected), unit=unit, action=action, evidence=evidence))
if COPY_POLICY == "remove_confirmed":
    clean = clean[clean.record_id != "R13"].copy()
    record("Extra export copy", 1, "row removed", "Remove R13", "Source confirms copy of R01")
if VERSION_POLICY == "verified_correction":
    clean = clean[clean.record_id != "R02"].copy()
    record("Superseded exam version", 1, "row removed", "Remove R02; retain R14", "Source verifies R14 correction")
clean["programme_raw"] = clean.programme
clean["programme"] = clean.programme.str.strip().replace({"biology":"Biology", "BIO":"Biology"})
record("Programme labels", (clean.programme != clean.programme_raw).sum(), "cells changed", "Apply approved mapping", "Dictionary; Bioengineering remains separate")
clean["entry_score_raw"] = clean.entry_score
score_text = clean.entry_score.str.strip().str.removesuffix("%")
clean["entry_score"] = pd.to_numeric(score_text, errors="coerce")
clean["score_status"] = np.select(
    [clean.entry_score_raw.eq(""), clean.entry_score.isna()],
    ["originally_missing", "unparsed_needs_source"], default="observed")
record("Percentage representation", clean.entry_score_raw.str.endswith("%").sum(), "cells converted", "Store numeric points", "Dictionary specifies 0–100 points")
record("Unparsed score", clean.score_status.eq("unparsed_needs_source").sum(), "values flagged", "Keep missing; retain raw text", "No verified exam value")
record("Original blank score", clean.score_status.eq("originally_missing").sum(), "values retained missing", "No imputation", "No source measurement")
clean["age_raw"] = clean.age
clean["age"] = pd.to_numeric(clean.age, errors="coerce")
invalid_age = ~clean.age.between(16,100)
record("Invalid age", invalid_age.sum(), "values flagged" if INVALID_AGE_POLICY=="flag_value" else "rows removed", INVALID_AGE_POLICY, "Dictionary; correct age unknown")
if INVALID_AGE_POLICY == "flag_value":
    clean.loc[invalid_age, "age"] = np.nan
else:
    clean = clean.loc[~invalid_age].copy()
clean["age_flag"] = clean.age.isna()
clean["updated_at"] = pd.to_datetime(clean.updated_at, format="%d/%m/%Y")
clean["phone"] = clean.phone.replace("", pd.NA)
clean = clean.sort_values("student_id").reset_index(drop=True)
cleaning_log = pd.DataFrame(log)
display(clean[["record_id","student_id","programme","age","entry_score","score_status"]])
display(cleaning_log)
print("Rows before/after:", len(raw), len(clean))
print("Scores missing after cleaning:", clean.entry_score.isna().sum())

### Your explanation
**Q4 · Interpret the result and challenge the shortcut (13 points).**
1. Did the results match your prediction? Explain any revision to your thinking.
2. Cite the record IDs removed and distinguish a **row removed** from a **value flagged**.
3. A colleague says “keep the latest row for every ID.” In this case, why is the retained version justified? If R14 were just a later upload with no verification, what should happen instead?
4. Two rows are for the same student in weeks 1 and 2. Should one be removed? State an appropriate key and row definition.
5. Try `"drop_student"` once: how does the row count change, and which usable score is lost? Return to your final policy and rerun the cleaning cell.

**Write here:** …

## 3 · Missingness changes who we see
This is a **separate fictional survey**, not the entry-score registry. Everyone in the roster was invited. Satisfaction ranges from 1 to 5. Group is known for everyone. No answer means unknown, not zero.

Before running: if Evening students answer less often, will the respondent mix necessarily match the invited mix? Write a prediction in Q5 first.

In [ ]:
survey_rng = np.random.default_rng(seed + 1)
n_day, n_evening = 20, 10
reply_day = int(survey_rng.integers(15,19))
reply_evening = int(survey_rng.integers(3,6))
survey = pd.DataFrame({"group": ["Day"]*n_day+["Evening"]*n_evening,
    "rating": [4.0]*reply_day+[np.nan]*(n_day-reply_day)+[2.0]*reply_evening+[np.nan]*(n_evening-reply_evening)})
summary = survey.groupby("group").rating.agg(invited="size", replies="count", observed_mean="mean")
summary["missing"] = summary.invited-summary.replies
summary["response_percent"] = (100*summary.replies/summary.invited).round(1)
summary["share_of_invited_percent"] = (100*summary.invited/len(survey)).round(1)
summary["share_of_replies_percent"] = (100*summary.replies/summary.replies.sum()).round(1)
display(summary)
print("Pooled observed mean:", round(survey.rating.mean(),3))
summary.response_percent.plot.bar(color=["#238b8d", "#dd9050"], ylim=(0,100), ylabel="Responses / invited (%)", rot=0)
plt.title("Who answered in your version?")
plt.show()

### Your explanation
**Q5 · Denominators and representation (10 points).**
- Prediction before running: **…**
- Report the missing fraction for each group using **counts and denominators**.
- Compare Evening's share of invitations with its share of replies. Who becomes less represented after dropping missing ratings?
- Write one accurate sentence reporting the pooled observed mean and valid n. Explain why it does not establish the all-student mean.
- Explain the difference between **0 hours of lab use**, **unknown hours**, and **not enrolled in the lab module**. Which students belong in the target population when summarizing hours for enrolled students?

**Write here:** …

### Three possible stories for the same survey
**Story A:** after everyone answers, a technical process independently loses each answer with equal probability, unrelated to group or rating.

**Story B:** Evening students have less time to answer. Within each group, reply probability does not depend on satisfaction.

**Story C:** within the Evening group, dissatisfied students are less likely to answer than satisfied students, even after accounting for known group.

These are hypothetical mechanisms; the observed response counts do not tell us which story actually occurred.

### Your explanation
**Q6 · Explain mechanisms, not just labels (10 points).**
1. Match A, B, C to MCAR, MAR, MNAR and justify each in an ordinary sentence.
2. Does the group difference in your table prove MAR? State exactly what must be true **within each group** for Story B.
3. Propose one practical follow-up that could help investigate nonresponse. Explain what would remain uncertain.
4. If both groups had identical response rates, would this guarantee an unbiased comparison? Give a counterexample.

**Write here:** …

## 4 · Four responses to missing values
Decide what each situation permits. Recovering a verified value is different from estimating it.

| Situation | Your choice: recover / keep missing / exclude for stated analysis / impute as a model assumption | Reason and what to report |
|---|---|---|
| A verified exam sheet for the exact student and exam contains the missing score | … | … |
| Programme is known; phone is blank; the question is programme counts | … | … |
| You explicitly report scores only for students with recorded scores | … | … |
| A separate demonstration fills blanks with the observed mean | … | … |

**Q7a (5 points):** complete this table. Explain why the fourth row does not create newly observed evidence. Keep your final registry unfilled.

In [ ]:
# Separate demonstration only: neither clean nor survey is overwritten.
example = pd.Series([2.0,4.0,np.nan,np.nan,np.nan])
filled_demo = example.fillna(example.mean())
display(pd.DataFrame({
    "treatment":["Observed only", "Mean-filled demonstration"],
    "mean":[example.mean(), filled_demo.mean()],
    "sample_SD":[example.std(), filled_demo.std()],
    "numeric_cells":[example.count(), filled_demo.count()],
    "genuinely_observed_answers":[2,2]
}).round(3))
# Hypothetical values for missing responses: sensitivity, NOT a cleaned dataset.
observed_total = survey.rating.sum()
missing_n = int(survey.rating.isna().sum())
sensitivity = pd.DataFrame({"assumed_mean_of_missing":[1.,2.,3.,4.,5.]})
sensitivity["hypothetical_all_student_mean"] = (observed_total + missing_n*sensitivity.assumed_mean_of_missing)/len(survey)
sensitivity["meets_target_3_5"] = sensitivity.hypothetical_all_student_mean >= 3.5
display(sensitivity.round(3))
print("Target: all-student satisfaction mean at least 3.5")

### Your explanation
**Q7b · What can we claim? (10 points).**
1. In the imputation demonstration, what changes and what stays the same? Why is “we now have five observed answers” false?
2. Cite two rows from your sensitivity table that lead to different target decisions. Are these recovered answers, or assumptions?
3. State the smallest and largest possible all-student mean on the 1–5 scale using the table. Is “the course definitely meets the target” defensible?
4. Write a cautious alternative conclusion and suggest what information would strengthen it. No formula derivation is needed.

**Write here:** …

## 5 · Flagged does not mean wrong
The code computes an IQR review rule for you. **You only need to use the output:** a flag marks a value worth investigating; it is not an instruction to delete.

This review table uses the original 12 students, before correcting the invalid age. The dictionary and source evidence remain essential.

In [ ]:
ages = pd.to_numeric(base.age)
q1, q3 = ages.quantile([0.25,0.75])
low, high = q1-1.5*(q3-q1), q3+1.5*(q3-q1)
review = pd.DataFrame({"student_id":base.student_id, "age":ages,
    "flagged_for_review": (ages<low)|(ages>high)})
print("Computed review limits:", round(low,2), "to", round(high,2), "years")
display(review[review.flagged_for_review])
# A separate fully verified cohort illustrates the consequences of deleting older students.
older_n = int(rng.integers(4,8))
older_withdrawals = older_n-1
other_n, other_withdrawals = 100-older_n, 6
impact = pd.DataFrame({"population":["All enrolled students","After excluding verified older students"],
    "students":[100,other_n], "withdrawals":[older_withdrawals+other_withdrawals,other_withdrawals]})
impact["withdrawal_percent"] = 100*impact.withdrawals/impact.students
display(impact.round(2))

### Your explanation
**Q8 · Same flag, different actions (10 points).**
1. Compare age 220 and your verified older age. Use the output and source evidence to justify different actions.
2. Does an age inside the review limits guarantee accuracy? Give a plausible recording error that could pass.
3. Quote the two withdrawal rates and their denominators. Did deleting valid older students improve the underlying outcome, or change who was counted?
4. If the intended population is all enrolled students, which rate should lead your report? What useful comparison could you still show?

**Write here:** …

## 6 · Audit, report, and transfer your understanding
Run the checks after returning to your final cleaning policy. A failed check is useful feedback: inspect your decision settings and rerun Part 2, then this cell. You do not need to rewrite the checks.

In [ ]:
checks = {
    "One row per student": clean.student_id.is_unique,
    "All expected students retained": set(clean.student_id)==set(ids),
    "IDs remain five-character text": clean.student_id.map(lambda x:isinstance(x,str) and len(x)==5).all(),
    "Only approved programmes": clean.programme.isin(["Biology","Bioengineering","Economics"]).all(),
    "Observed scores within 0–100": clean.entry_score.dropna().between(0,100).all(),
    "Observed ages within dictionary range": clean.age.dropna().between(16,100).all(),
    "Unresolved scores remain missing": clean.loc[clean.score_status!="observed","entry_score"].isna().all(),
    "Raw export unchanged": raw.equals(raw_snapshot),
    "Row removals reconciled": len(raw)-len(clean)==sum(x["affected"] for x in log if x["unit"] in ["row removed","rows removed"]),
}
display(pd.DataFrame({"check":checks.keys(),"passed":checks.values()}))
print("Final rows:",len(clean),"| Observed scores:",clean.entry_score.count(),"| Missing scores:",clean.entry_score.isna().sum())
print("Observed entry-score mean:",round(clean.entry_score.mean(),2),"points out of 100")
assert all(checks.values()), "Review failed checks and your policies; rerun Part 2 and then this cell."
print("Structural checks passed. Accuracy still requires source evidence.")

### Your explanation
**Q9 · Final report and audit (8 points).**
Write 100–150 words for a reader who has not opened your notebook. Include:
- your version code; target population; raw/final row counts;
- key actions with evidence, distinguishing row removal from value flagging;
- observed entry-score mean, its unit, valid n, and missing count;
- one limitation about missingness and one about accuracy.

Then cite **two entries** in the generated cleaning log. Explain why passing every check does not prove that every stored score matches the actual exam.

**Write here:** …

In [ ]:
challenge = [
    "An analyst replaces every missing phone with 0 and deletes students with missing scores before counting programmes. Evaluate BOTH decisions for the programme-count question.",
    "A new weekly-assessment export has two rows per student, one per week. An analyst reuses the current-registry rule and keeps only the latest. Explain what is lost and define the correct key.",
    "An analyst merges Bioengineering into Biology because both start with Bio, then reports that all validity checks pass. Explain the semantic error and why a check can miss it."
][seed % 3]
print("YOUR CHANGED-CASE CHALLENGE:", challenge)

### Your explanation
**Q10 · Independent transfer (7 points).** Answer your displayed challenge in 60–100 words. Give (a) the problem, (b) a better action and supporting evidence needed, and (c) a consequence for the conclusion. Finally name one answer in this lab you changed after inspecting evidence, or explain why you retained your initial answer.

**Write here:** …

## Export and submission
Run the cell below to create a ZIP with your **cleaned registry CSV**, **cleaning log CSV**, and original synthetic export. The ZIP does **not** contain the notebook: download your completed `.ipynb` separately with all answers and outputs.

**Submit:** completed notebook + generated ZIP through the Lab 3 submission route specified by your instructor/course page. Do not replace an existing course submission subject or deadline with a guessed one.

Before submitting:
- [ ] My own student ID, name, and class group are entered.
- [ ] I reran all cells after selecting my final policy; checks pass.
- [ ] Q1–Q10 and the four-responses table have my explanations and relevant numbers.
- [ ] Unknown values remain missing in the final registry.
- [ ] My downloaded notebook contains outputs and written answers; the ZIP opens.

**Lecture alignment:** Week 3 *Data Cleaning and Quality*, student slides dated 1 October 2026: quality/dictionary (6–15); duplicates (16–20); missingness and responses (23–39); unusual values (40–44); audit/assertions/reporting (46–55). This lab implements the lecture instruction to keep missing values unfilled. It uses built-in synthetic data so no external dataset download is needed.

In [ ]:
assert all(checks.values()), "Complete the checks before exporting."
assert STUDENT_ID != "DEMO123", "Enter your own ID, then rerun all cells before exporting."
assert STUDENT_NAME != "Your name" and CLASS_GROUP != "Your class group", "Complete your name and class group."
zip_name = f"{STUDENT_ID}_Lab3_Data_and_Log.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    z.writestr(f"{STUDENT_ID}_Lab3_Cleaned_Registry.csv", clean.to_csv(index=False))
    z.writestr(f"{STUDENT_ID}_Lab3_Cleaning_Log.csv", cleaning_log.to_csv(index=False))
    z.writestr(f"{STUDENT_ID}_Lab3_Raw_Export.csv", raw.to_csv(index=False))
    z.writestr("VERSION.txt", f"Student: {STUDENT_ID}\nName: {STUDENT_NAME}\nGroup: {CLASS_GROUP}\nVersion: {seed:08x}\nSynthetic teaching data. Missing values intentionally unfilled.\n")
print("Created", zip_name, "— also download your completed notebook separately.")
try:
    from google.colab import files
    files.download(zip_name)
except ImportError:
    display(FileLink(zip_name))